# PROTEUS Quickstart

Score a protein for conformational plasticity from pre-extracted embeddings.
No GPU or SimpleFold required for this notebook.


In [ ]:
from pathlib import Path
import proteus

# Load a pre-computed embedding NPZ file
emb = proteus.load_embeddings("path/to/protein.npz")
print(f"Protein: {emb.protein_id}  L={emb.seq_emb.shape[0]}  K={len(emb.struct_embs)}")

In [ ]:
# Compute the primary PROTEUS score
score = proteus.l2_delta_max(emb.seq_emb, emb.struct_embs)
print(f"l2_delta_max: {score:.3f}")

# Or compute all features at once
features = proteus.compute_all_features(emb.seq_emb, emb.struct_embs)
for k, v in features.items():
    print(f"  {k}: {v:.4f}")

In [ ]:
# Score many proteins from a directory
import pandas as pd

records = []
for path in Path("embeddings/").glob("*.npz"):
    emb = proteus.load_embeddings(path)
    if emb is None:
        continue
    records.append({"protein_id": emb.protein_id,
                    **proteus.compute_all_features(emb.seq_emb, emb.struct_embs)})

df = pd.DataFrame(records)
df.sort_values("l2_delta_max", ascending=False).head(10)